# ModelDeploymentPrep — final fixed version

This notebook is self-contained and robust against the previous errors:

- `best_model.joblib` missing
- `dict object has no attribute transform`
- `X_inference is not defined`
- `pd is not defined`
- `PROCESSED_DIR is not defined`
- Giskard section failing when run separately

Run order:
1. Run `DataPreprocessing_final_no_skip.ipynb` first.
2. Then run this notebook with `Kernel → Restart Kernel → Run All Cells`.

The notebook uses processed features from `data/processed/` and creates a fallback model if a trained model artifact is missing.


In [10]:
# Core imports and paths

from pathlib import Path
import time
import os
import pickle
import warnings

import numpy as np
import pandas as pd
import joblib

from sklearn.metrics import mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor

warnings.filterwarnings("ignore")

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
REPORTS_DIR = PROJECT_ROOT / "reports"
MODELS_DIR = PROJECT_ROOT / "models"
BEST_MODEL_DIR = MODELS_DIR / "best"

for directory in [DATA_DIR, PROCESSED_DIR, REPORTS_DIR, MODELS_DIR, BEST_MODEL_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Processed dir:", PROCESSED_DIR)
print("Reports dir:", REPORTS_DIR)
print("Models dir:", MODELS_DIR)
print("Best model dir:", BEST_MODEL_DIR)


Project root: /home/jovyan/project_ivanov
Processed dir: /home/jovyan/project_ivanov/data/processed
Reports dir: /home/jovyan/project_ivanov/reports
Models dir: /home/jovyan/project_ivanov/models
Best model dir: /home/jovyan/project_ivanov/models/best


In [11]:
# Load processed datasets

def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

required_files = {
    "X_train": PROCESSED_DIR / "X_train.csv",
    "X_test": PROCESSED_DIR / "X_test.csv",
    "y_train": PROCESSED_DIR / "y_train.csv",
    "y_test": PROCESSED_DIR / "y_test.csv",
}

missing = [str(path) for path in required_files.values() if not path.exists()]
if missing:
    raise FileNotFoundError(
        "Run DataPreprocessing_final_no_skip.ipynb first. Missing files: " + ", ".join(missing)
    )

X_train = pd.read_csv(required_files["X_train"])
X_test = pd.read_csv(required_files["X_test"])
y_train = pd.read_csv(required_files["y_train"]).iloc[:, 0]
y_test = pd.read_csv(required_files["y_test"]).iloc[:, 0]

# This variable is always defined before Giskard or inference cells.
X_inference = X_test.copy()

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("X_inference:", X_inference.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)


X_train: (40000, 273)
X_test: (10000, 273)
X_inference: (10000, 273)
y_train: (40000,)
y_test: (10000,)


In [12]:
# Load existing model or train fallback model

best_model_path = BEST_MODEL_DIR / "best_model.joblib"

if best_model_path.exists():
    model = joblib.load(best_model_path)
    print("Loaded existing model:", best_model_path)
else:
    print("best_model.joblib was not found. Training fallback RandomForestRegressor...")
    model = RandomForestRegressor(
        n_estimators=100,
        max_depth=18,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1
    )
    model.fit(X_train, y_train)
    joblib.dump(model, best_model_path)
    print("Saved fallback model:", best_model_path)

pred = model.predict(X_inference)
print(f"RMSE: ${rmse(y_test, pred):,.2f}")
print(f"R²: {r2_score(y_test, pred):.4f}")


Loaded existing model: /home/jovyan/project_ivanov/models/best/best_model.joblib
RMSE: $4,644.56
R²: 0.7878


## 1. Inference speed

In [13]:
# Inference speed check

if "X_inference" not in globals():
    X_inference = X_test.copy()

_ = model.predict(X_inference.head(5))

start = time.time()
pred = model.predict(X_inference)
inference_time = time.time() - start

print(f"RMSE: ${rmse(y_test, pred):,.2f}")
print(f"R²: {r2_score(y_test, pred):.4f}")
print(f"Total inference time: {inference_time:.4f} sec")
print(f"Average inference time per row: {inference_time / len(X_inference) * 1000:.6f} ms")


RMSE: $4,644.56
R²: 0.7878
Total inference time: 0.0658 sec
Average inference time per row: 0.006581 ms


## 2. Model saving/loading comparison

In [14]:
# Compare joblib and pickle saving/loading

if "X_inference" not in globals():
    X_inference = X_test.copy()

comparison_rows = []

# Joblib
joblib_path = MODELS_DIR / "best_model_joblib.joblib"
start = time.time()
joblib.dump(model, joblib_path)
save_time = time.time() - start

start = time.time()
loaded_joblib = joblib.load(joblib_path)
load_time = time.time() - start

start = time.time()
pred_joblib = loaded_joblib.predict(X_inference)
predict_time = time.time() - start

comparison_rows.append({
    "method": "joblib",
    "file_size_mb": os.path.getsize(joblib_path) / 1e6,
    "save_time_sec": save_time,
    "load_time_sec": load_time,
    "predict_time_sec": predict_time,
    "rmse": rmse(y_test, pred_joblib),
    "r2": r2_score(y_test, pred_joblib)
})

# Pickle
pickle_path = MODELS_DIR / "best_model_pickle.pkl"
start = time.time()
with open(pickle_path, "wb") as f:
    pickle.dump(model, f)
save_time = time.time() - start

start = time.time()
with open(pickle_path, "rb") as f:
    loaded_pickle = pickle.load(f)
load_time = time.time() - start

start = time.time()
pred_pickle = loaded_pickle.predict(X_inference)
predict_time = time.time() - start

comparison_rows.append({
    "method": "pickle",
    "file_size_mb": os.path.getsize(pickle_path) / 1e6,
    "save_time_sec": save_time,
    "load_time_sec": load_time,
    "predict_time_sec": predict_time,
    "rmse": rmse(y_test, pred_pickle),
    "r2": r2_score(y_test, pred_pickle)
})

saving_comparison = pd.DataFrame(comparison_rows)
saving_comparison_path = REPORTS_DIR / "model_saving_comparison.csv"
saving_comparison.to_csv(saving_comparison_path, index=False)

print("Saved comparison to:", saving_comparison_path)
saving_comparison


Saved comparison to: /home/jovyan/project_ivanov/reports/model_saving_comparison.csv


,method,file_size_mb,save_time_sec,load_time_sec,predict_time_sec,rmse,r2
0,joblib,51.592193,0.204062,0.041018,0.066880,4644.562839,0.787809
1,pickle,51.588648,0.177860,0.016561,0.066278,4644.562839,0.787809


## 3. Optional Giskard scan

This cell is fully self-contained. It re-imports libraries and re-creates all paths, so it can be run separately without `pd`, `PROCESSED_DIR`, or `X_inference` errors.

If Giskard fails because of environment compatibility, the notebook will not crash; it will write a small fallback HTML report instead.


In [ ]:
# Robust optional Giskard scan

from pathlib import Path
import pandas as pd
import numpy as np
import joblib

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
REPORTS_DIR = PROJECT_ROOT / "reports"
MODELS_DIR = PROJECT_ROOT / "models"
BEST_MODEL_DIR = MODELS_DIR / "best"

REPORTS_DIR.mkdir(parents=True, exist_ok=True)
BEST_MODEL_DIR.mkdir(parents=True, exist_ok=True)

X_test_for_scan = pd.read_csv(PROCESSED_DIR / "X_test.csv")
y_test_for_scan = pd.read_csv(PROCESSED_DIR / "y_test.csv").iloc[:, 0]
model_for_scan = joblib.load(BEST_MODEL_DIR / "best_model.joblib")

X_inference_for_scan = X_test_for_scan.copy()

try:
    import giskard
    from giskard import Model, Dataset, scan

    giskard_X = X_inference_for_scan.head(1000).copy()
    giskard_y = y_test_for_scan.head(1000).reset_index(drop=True)

    def prediction_function(df):
        return model_for_scan.predict(df)

    giskard_model = Model(
        model=prediction_function,
        model_type="regression",
        name="Vehicle price prediction model",
        feature_names=list(giskard_X.columns)
    )

    giskard_dataset = Dataset(
        df=pd.concat(
            [giskard_X.reset_index(drop=True), giskard_y.rename("sellingprice")],
            axis=1
        ),
        target="sellingprice",
        name="Vehicle sales processed test dataset"
    )

    scan_report = scan(giskard_model, giskard_dataset)
    report_path = REPORTS_DIR / "giskard_scan_report.html"
    scan_report.to_html(str(report_path))
    print("Saved Giskard report:", report_path)

except ModuleNotFoundError:
    report_path = REPORTS_DIR / "giskard_scan_report.html"
    report_path.write_text(
        """<!DOCTYPE html>
<html>
<head><meta charset="utf-8"><title>Giskard Scan Report</title></head>
<body>
<h1>Giskard Scan Report</h1>
<p>Giskard is not installed in this environment, so the scan was skipped.</p>
<p>The deployment preparation notebook still completed model loading, inference speed analysis, and model saving/loading comparison.</p>
</body>
</html>
""",
        encoding="utf-8"
    )
    print("Giskard is not installed. Created fallback report:", report_path)

except Exception as e:
    report_path = REPORTS_DIR / "giskard_scan_report.html"
    report_path.write_text(
        f"""<!DOCTYPE html>
<html>
<head><meta charset="utf-8"><title>Giskard Scan Report</title></head>
<body>
<h1>Giskard Scan Report</h1>
<p>Giskard scan was skipped because of this error:</p>
<pre>{type(e).__name__}: {e}</pre>
<p>The deployment preparation notebook still completed model loading, inference speed analysis, and model saving/loading comparison.</p>
</body>
</html>
""",
        encoding="utf-8"
    )
    print("Giskard scan was skipped, but fallback report was created:", report_path)
    print(type(e).__name__ + ":", e)


2026-06-16 19:34:28,941 pid:235 MainThread giskard.models.automodel INFO     Your 'prediction_function' is successfully wrapped by Giskard's 'PredictionFunctionModel' wrapper class.
2026-06-16 19:34:28,977 pid:235 MainThread giskard.datasets.base INFO     Your 'pandas.DataFrame' is successfully wrapped by Giskard's 'Dataset' wrapper class.
2026-06-16 19:34:29,013 pid:235 MainThread giskard.datasets.base INFO     Casting dataframe columns from {'num__year': 'float64', 'num__condition': 'float64', 'num__odometer': 'float64', 'cat__make__Acura': 'float64', 'cat__make__Audi': 'float64', 'cat__make__BMW': 'float64', 'cat__make__Buick': 'float64', 'cat__make__Cadillac': 'float64', 'cat__make__Chevrolet': 'float64', 'cat__make__Chrysler': 'float64', 'cat__make__Dodge': 'float64', 'cat__make__FIAT': 'float64', 'cat__make__Ford': 'float64', 'cat__make__GMC': 'float64', 'cat__make__HUMMER': 'float64', 'cat__make__Honda': 'float64', 'cat__make__Hyundai': 'float64', 'cat__make__Infiniti': 'float64

## 4. Summary

Generated artifacts:

- `models/best/best_model.joblib`
- `models/best_model_joblib.joblib`
- `models/best_model_pickle.pkl`
- `reports/model_saving_comparison.csv`
- `reports/giskard_scan_report.html`

The Giskard report is real if Giskard runs successfully; otherwise it is a fallback HTML report explaining why the scan was skipped.
